# Étape 2 — Rémi : qualité des données

## Objectif

Vérifier que le jeu de données est suffisamment propre avant de poursuivre l'analyse exploratoire : aperçu, valeurs manquantes, doublons, incohérences et valeurs atypiques.


In [3]:
import pandas as pd

from load_data import load_prepared

df = load_prepared()
df.head()

quality_summary = pd.DataFrame({
    "lignes": [len(df)],
    "colonnes": [df.shape[1]],
    "doublons_complets": [int(df.duplicated().sum())],
    "notes_hors_1_5": [int((~df["rating"].between(1, 5)).sum())],
    "user_id_manquants": [int(df["user_id"].isna().sum())],
    "item_id_manquants": [int(df["item_id"].isna().sum())],
    "titres_manquants": [int(df["title"].isna().sum())],
})
display(quality_summary)

missing = (
    df.isna().sum()
      .rename("n_manquants")
      .to_frame()
      .assign(pourcentage=lambda x: (100 * x["n_manquants"] / len(df)).round(2))
      .query("n_manquants > 0")
      .sort_values("n_manquants", ascending=False)
)
display(missing)

incoherences = pd.DataFrame({
    "controle": [
        "notes hors de l'intervalle [1, 5]",
        "âges hors de l'intervalle plausible [1, 120]",
        "années de sortie avant 1888 ou futures",
        "timestamps invalides",
    ],
    "effectif": [
        int((~df["rating"].between(1, 5)).sum()),
        int((df["age"].notna() & ~df["age"].between(1, 120)).sum()),
        int((df["release_year"].notna() & ~df["release_year"].between(1888, pd.Timestamp.now().year)).sum()),
        int(df["datetime"].isna().sum()),
    ],
})
display(incoherences)

rating_counts = df["rating"].value_counts().sort_index().rename_axis("rating").to_frame("effectif")
display(rating_counts)

user_activity = df.groupby("user_id").size()
movie_activity = df.groupby("item_id").size()

def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).sum())

activity_outliers = pd.DataFrame({
    "niveau": ["utilisateur", "film"],
    "observations": [len(user_activity), len(movie_activity)],
    "candidats_outliers_iqr": [
        iqr_outlier_count(user_activity),
        iqr_outlier_count(movie_activity),
    ],
})
display(activity_outliers)

print("Conclusion qualité :")
print("- Les notes doivent rester comprises entre 1 et 5.")
print("- Les lignes dupliquées et les identifiants manquants sont surveillés avant l'analyse.")
print("- Les valeurs manquantes de release_date et release_year correspondent à des dates de sortie absentes du catalogue.")
print("- Les outliers d'activité sont des candidats à analyser : ils peuvent refléter des utilisateurs très actifs ou des films populaires, pas nécessairement des erreurs.")


,lignes,colonnes,doublons_complets,notes_hors_1_5,user_id_manquants,item_id_manquants,titres_manquants
0,100000,38,0,0,0,0,0


,n_manquants,pourcentage
video_release_date,100000,100.00
imdb_url,13,0.01
release_date,9,0.01
release_year,9,0.01


,controle,effectif
0,"notes hors de l'intervalle [1, 5]",0
1,"âges hors de l'intervalle plausible [1, 120]",0
2,années de sortie avant 1888 ou futures,0
3,timestamps invalides,0


,effectif
rating,
1,6110
2,11370
3,27145
4,34174
5,21201


,niveau,observations,candidats_outliers_iqr
0,utilisateur,943,42
1,film,1682,127


Conclusion qualité :
- Les notes doivent rester comprises entre 1 et 5.
- Les lignes dupliquées et les identifiants manquants sont surveillés avant l'analyse.
- Les valeurs manquantes de release_date et release_year correspondent à des dates de sortie absentes du catalogue.
- Les outliers d'activité sont des candidats à analyser : ils peuvent refléter des utilisateurs très actifs ou des films populaires, pas nécessairement des erreurs.
